# Fast Chow Polynomial Computation for $K_n$

This implementation calculates the Chow polynomial $H_{K_n}(y)$ using the **Integer Partition Recurrence**.

## 1. The Formula

$$H_{K_{n+1}}(y) = (1 - ny) H_{K_n}(y) + y \sum_{\lambda \vdash n} C_\lambda \cdot |\lambda| \cdot H_{K_{|\lambda|}}(y) \cdot H_{\oplus}(\lambda, y)$$

Where:
* $\lambda \vdash n$: An integer partition of $n$.
* $|\lambda|$: The number of parts in $\lambda$.
* $C_\lambda$: The number of set partitions of $\{1,\dots,n\}$ with block sizes $\lambda$.
* $H_{\oplus}(\lambda, y)$: The Chow polynomial of the direct sum $K_{\lambda_1} \oplus K_{\lambda_2} \oplus \dots$

## 2. Optimization Strategy

To make this computationally feasible for $N \ge 9$:

1. **Direct Sum via Pielasa [arXiv:2511.10746 Thm 1.7]:** For $M = K_a$, $N = K_{b_1} \oplus \cdots \oplus K_{b_r}$, we compute $H_{M \oplus N}$ recursively:
   $$H_{M \oplus N}(x) = H_M(x) H_N(x) + x \sum_{F < M,\, G < N} C_F \, C_G \cdot H_{M|F \oplus N|G}(x) \cdot H_{M^F}(x) \cdot H_{N^G}(x)$$
   where the sum runs over *strictly proper* flats $F < M$, $G < N$; $C_F, C_G$ count set partitions with the given block sizes; $M^F, N^G$ are contractions (yielding cliques); $M|F \oplus N|G$ is the restriction (direct sum of smaller cliques). For $K_n$, flats correspond to partitions of $[n]$, so we iterate over partitions with $\ge 2$ blocks and compute $H_{N^G}$ recursively (not as a product).

2. **Formula in terms of partitions (braids):** For shape $\lambda = (a_1, a_2, \ldots, a_r)$ with $a_1 \le \cdots \le a_r$, writing $H(\lambda) = H_{K_{a_1} \oplus \cdots \oplus K_{a_r}}$:
   $$\begin{align}
   H(a_1,\ldots,a_r) &= H(a_1) \cdot H(a_2,\ldots,a_r) \\
   &\quad + x \sum_{\substack{\pi \vdash a_1,\, |\pi| \ge 2 \\ (\pi_2,\ldots,\pi_r):\, \pi_i \vdash a_i,\, \text{not all } |\pi_i|=1}} C(\pi,a_1) \prod_{i=2}^r C(\pi_i,a_i) \\
   &\quad \cdot H(\pi \sqcup \pi_2 \sqcup \cdots \sqcup \pi_r) \cdot H(|\pi|) \cdot H(|\pi_2|,\ldots,|\pi_r|)
   \end{align}$$
   where $\pi \sqcup \pi_2 \sqcup \cdots \sqcup \pi_r$ denotes the sorted concatenation of block sizes (restriction shape), and $H(|\pi_2|,\ldots,|\pi_r|)$ is computed recursively (contraction of $N$ by flat $G$).

3. **Shape Caching:** Cache invariants by sorted tuple of clique sizes in `chowBraid.json`.

In [25]:
import sys
import os

current_dir = os.path.abspath(os.getcwd())
if os.path.exists(os.path.join(current_dir, "conclusion", "explorations on complete graphs")):
    project_root = current_dir
else:
    project_root = os.path.dirname(os.path.dirname(current_dir))
sys.path.insert(0, project_root)
sys.path.insert(0, os.path.join(project_root, "the-machine"))

from sage.all import *
from HM_module import *
from custom_matroid_functions import *

In [ ]:
import json
from collections import Counter

P = LaurentPolynomialRing(QQ, 'x')
x = P.gen()

# --- Cache paths (chowBraid.json) ---
_expl_dir = os.path.join(os.getcwd(), "conclusion", "explorations on complete graphs")
CHOWBRAID_CACHE_PATH = os.path.join(_expl_dir, "chowBraid.json") if os.path.exists(_expl_dir) else os.path.join(os.getcwd(), "chowBraid.json")

# --- Global Caches ---
_Kn_cache = {}
_invariant_cache = {}

def _parse_poly(s):
    if s == "1" or s == 1:
        return P(1)
    return P(str(s))

def _load_chowBraid_cache():
    """Load _Kn_cache and _invariant_cache from chowBraid.json."""
    if os.path.exists(CHOWBRAID_CACHE_PATH):
        try:
            with open(CHOWBRAID_CACHE_PATH, 'r') as f:
                data = json.load(f)
            kn = {int(k): _parse_poly(v) for k, v in data.get("H_Kn", {}).items()}
            inv = {tuple(json.loads(k)): _parse_poly(v) for k, v in data.get("H_direct_sum", {}).items()}
            return kn, inv
        except (json.JSONDecodeError, ValueError) as e:
            print(f"Warning: Could not load chowBraid cache: {e}")
    return {}, {}

def _save_chowBraid_cache():
    """Save _Kn_cache and _invariant_cache to chowBraid.json."""
    try:
        h_kn = {str(int(k)): str(v) for k, v in _Kn_cache.items()}
        h_ds = {json.dumps([int(x) for x in k]): str(v) for k, v in _invariant_cache.items()}
        with open(CHOWBRAID_CACHE_PATH, 'w') as f:
            json.dump({"H_Kn": h_kn, "H_direct_sum": h_ds}, f, indent=2)
    except Exception as e:
        print(f"Warning: Could not save chowBraid cache: {e}")

def C_lambda(lam, n):
    """Number of set partitions of {1..n} with block sizes lam."""
    mult = Counter(lam)
    denom = 1
    for j, mj in mult.items():
        denom *= factorial(int(j)) ** int(mj) * factorial(int(mj))
    return factorial(int(n)) // denom

def _get_H_Kn(n, kn):
    """Return H_{K_n}; compute via get_chow_polynomial if not in cache."""
    if n not in kn:
        kn[n] = P(get_chow_polynomial(matroids.CompleteGraphic(n)))
    return kn[n]

def get_invariant(shape, kn_ref=None):
    """Chow polynomial for K_λ₁ ⊕ K_λ₂ ⊕ ... using Pielasa Thm 1.7 (strictly proper flats).
    Recursively computes H(N/G) instead of multiplying. Strips K_1 components."""
    import itertools
    shape = tuple(sorted([int(s) for s in shape if s > 1]))
    if not shape:
        return P(1)
    kn = kn_ref if kn_ref is not None else _Kn_cache
    if shape in _invariant_cache:
        return _invariant_cache[shape]
    if len(shape) == 1:
        return _get_H_Kn(shape[0], kn)
    a1, rest = shape[0], shape[1:]
    term_base = _get_H_Kn(a1, kn) * get_invariant(rest, kn)
    term_sum = P(0)
    for lam in Partitions(a1):
        lam_list = list(lam)
        if len(lam_list) < 2:
            continue
        c_M = C_lambda(lam_list, a1)
        h_cont_M = _get_H_Kn(len(lam_list), kn)
        comp_parts = [[list(p) for p in Partitions(r)] for r in rest]
        for parts_tuple in itertools.product(*comp_parts):
            if all(len(p) == 1 for p in parts_tuple):
                continue
            c_N = 1
            for i, p in enumerate(parts_tuple):
                c_N *= C_lambda(p, rest[i])
            contr_shape_N = tuple(len(p) for p in parts_tuple)
            h_cont_N = get_invariant(contr_shape_N, kn)
            combined_restr = list(lam_list) + [b for p in parts_tuple for b in p]
            h_restr = get_invariant(tuple(combined_restr), kn)
            term_sum += x * (c_M * c_N) * h_restr * (h_cont_M * h_cont_N)
    res = term_base + term_sum
    _invariant_cache[shape] = res
    return res

def compute_sequence(n, use_cache=True):
    """Compute H_K1 through H_KN using the integer partition recurrence.
    Set use_cache=False to ignore chowBraid.json and recompute from scratch (e.g. if cache was corrupted)."""
    global _Kn_cache, _invariant_cache
    if use_cache:
        _Kn_cache, _invariant_cache = _load_chowBraid_cache()
    else:
        _Kn_cache, _invariant_cache = {}, {}
    _Kn_cache[1] = P(1)
    _invariant_cache[(1,)] = P(1)
    term_base = (1 - n*x) * _Kn_cache[n]
    term_sum = P(0)
    for lam in Partitions(n):
        lam_list = list(lam)
        k = len(lam_list)
        C = C_lambda(lam_list, n)
        H_cont = _Kn_cache[k]
        H_restr = get_invariant(tuple(int(x) for x in lam_list))
        term_sum += x * C * k * H_cont * H_restr
    H_next = term_base + term_sum
    _Kn_cache[n+1] = H_next
    _invariant_cache[(n+1,)] = H_next
    print(f"H_K{n+1} computed. {H_next}")
    return _Kn_cache[n]

In [27]:
N_TARGET = 26
final_poly = compute_sequence(N_TARGET)  # use_cache=False to avoid corrupted cache
print(f"\nH_K{N_TARGET}:", final_poly)

H_K27 computed. 1 + 545717047936059989037*x + 15931452115029988538266123545*x^2 + 514390970606677088442428360326098*x^3 + 840796404537800411710111003244783886*x^4 + 243836243422452119266227176304416788842*x^5 + 21534426290007868448245921124979748673403*x^6 + 764357960943967247364055534640098730253578*x^7 + 12800206361131443937858466378354018150974773*x^8 + 111721625217878965140490674601767669463780713*x^9 + 541841726977816887449433766874674400139267213*x^10 + 1521390710770602439276282605905956335781119843*x^11 + 2534642722496509322726448868055637826022446428*x^12 + 2534642722496509322726448868055637826022446428*x^13 + 1521390710770602439276282605905956335781119843*x^14 + 541841726977816887449433766874674400139267213*x^15 + 111721625217878965140490674601767669463780713*x^16 + 12800206361131443937858466378354018150974773*x^17 + 764357960943967247364055534640098730253578*x^18 + 21534426290007868448245921124979748673403*x^19 + 243836243422452119266227176304416788842*x^20 + 8407964045378004

KeyError: <function numerical_approx at 0x1075244a0>